In [0]:
DESCRIBE HISTORY retail_lakehouse.silver.sales;

In [0]:
INSERT INTO retail_lakehouse.silver.sales
VALUES
(
    9999,      -- TransactionID
    1,         -- CustomerID
    1001,      -- ProductID
    100,       -- StoreID
    2,         -- Quantity
    CURRENT_DATE()
);


In [0]:

SELECT
    _change_type,
    _commit_version,
    TransactionID,
    CustomerID,
    ProductID,
    StoreID,
    Quantity
FROM table_changes(
    'retail_lakehouse.silver.sales',
    17
)
ORDER BY _commit_version DESC;

In [0]:
MERGE INTO retail_lakehouse.gold.fact_sales t

USING (

    SELECT
        s.TransactionID,
        c.CustomerSK,
        p.ProductSK,
        st.StoreSK,
        s.Quantity,
        s.Quantity * p.UnitPrice AS Amount,
        s.TxnDate

    FROM table_changes('retail_lakehouse.silver.sales', 17) s

    JOIN retail_lakehouse.gold.dim_customer c
        ON s.CustomerID = c.CustomerID
        AND c.IsActive = TRUE

    JOIN retail_lakehouse.gold.dim_product p
        ON s.ProductID = p.ProductID

    JOIN retail_lakehouse.gold.dim_store st
        ON s.StoreID = st.StoreID

    WHERE s._change_type = 'insert'

) src

ON t.TransactionID = src.TransactionID

WHEN NOT MATCHED THEN
INSERT (
    SalesSK,
    TransactionID,
    CustomerSK,
    ProductSK,
    StoreSK,
    Quantity,
    Amount,
    TxnDate
)

VALUES (
    monotonically_increasing_id(),
    src.TransactionID,
    src.CustomerSK,
    src.ProductSK,
    src.StoreSK,
    src.Quantity,
    src.Amount,
    src.TxnDate
);

In [0]:
SELECT *
FROM retail_lakehouse.gold.fact_sales
ORDER BY TransactionID DESC;

**SCD type 2 working**

In [0]:
SELECT CustomerID, IsActive
FROM retail_lakehouse.gold.dim_customer
WHERE CustomerID = 1;

SELECT
    CustomerID,
    StartDate,
    EndDate
FROM retail_lakehouse.gold.dim_customer
WHERE CustomerID = 1;

In [0]:
--VERIFY FACT TABLE UPDATED

SELECT *
FROM retail_lakehouse.gold.fact_sales
WHERE TransactionID = 9999;

--VALIDATE INCREMENTAL LOAD
--Verify only new rows inserted

SELECT COUNT(*)
FROM retail_lakehouse.gold.fact_sales;

--DUPLICATE VALIDATION

SELECT
    TransactionID,
    COUNT(*)
FROM retail_lakehouse.gold.fact_sales
GROUP BY TransactionID
HAVING COUNT(*) > 1;
-- EXPECTED RESULT:
-- 0 duplicate rows


--VERIFY FULL vs INCREMENTAL
-- Day 1:
-- Full dataset loaded
-- Day 2:
-- Only TransactionID 9999 inserted

SELECT *
FROM retail_lakehouse.gold.fact_sales
ORDER BY TransactionID DESC;